In [ ]:
import re
from pathlib import Path

from google.colab import drive

WHEEL_DIRECTORY = Path("/content/drive/MyDrive/data/jlens-reasoning/wheels")
REQUIREMENTS = WHEEL_DIRECTORY / "requirements-colab.txt"
COMMIT_FILE = WHEEL_DIRECTORY / "project-commit.txt"
DIRTY_FILE = WHEEL_DIRECTORY / "project-dirty.txt"

drive.mount("/content/drive")

if not COMMIT_FILE.is_file():
    raise RuntimeError(f"Missing project commit marker: {COMMIT_FILE}")
PROJECT_COMMIT = COMMIT_FILE.read_text(encoding="utf-8").strip()
if re.fullmatch(r"[0-9a-f]{40}", PROJECT_COMMIT) is None:
    raise RuntimeError("Project commit marker is invalid")
if not DIRTY_FILE.is_file():
    raise RuntimeError(f"Missing project dirty marker: {DIRTY_FILE}")
dirty_value = DIRTY_FILE.read_text(encoding="utf-8").strip()
if dirty_value not in {"true", "false"}:
    raise RuntimeError("Project dirty marker is invalid")
PROJECT_WORKING_TREE_DIRTY = dirty_value == "true"

wheels = sorted(WHEEL_DIRECTORY.glob("jlens_reasoning-*.whl"))
if not REQUIREMENTS.is_file():
    raise RuntimeError(f"Missing locked requirements: {REQUIREMENTS}")
if len(wheels) != 1:
    raise RuntimeError(
        f"Expected exactly one project wheel in {WHEEL_DIRECTORY}, found {len(wheels)}"
    )

wheel = wheels[0]
print(f"Installing locked environment from {REQUIREMENTS}")
%pip install -qq --disable-pip-version-check --requirement {REQUIREMENTS}
print(f"Installing project wheel {wheel.name}")
%pip install -qq --disable-pip-version-check --force-reinstall --no-deps {wheel}
print("Colab project installation complete")

del COMMIT_FILE, DIRTY_FILE, REQUIREMENTS, WHEEL_DIRECTORY, dirty_value, wheel, wheels

In [ ]:
from jlens_reasoning.environments.colab import initialize_colab

context = initialize_colab(enable_wandb=False, require_cuda=True)
context

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from jlens_reasoning.evaluation import evaluate_paper_binary

OUTPUT_DIR = context.runs_dir / "flenqa-accuracy"
RESULT_PATH = OUTPUT_DIR / "results.parquet"
MODEL_OUTPUT_PATH = (
    context.runs_dir / "flenqa-full-run" / "model_outputs.parquet"
)
LENGTHS = (250, 500, 1000, 2000, 3000)
EXPECTED_UNIQUE_COUNTS = {
    250: 300,
    500: 2_368,
    1000: 2_394,
    2000: 2_400,
    3000: 2_400,
}

In [ ]:
model_outputs = pq.read_table(MODEL_OUTPUT_PATH)
assert model_outputs.num_rows == 9_862
assert "generated_text" in model_outputs.column_names
model_outputs.schema

In [ ]:
records = model_outputs.to_pylist()
verdicts = []
correctness = []
for record in records:
    evaluation = evaluate_paper_binary(
        record["generated_text"], expected=record["label"]
    )
    verdicts.append(evaluation.verdict)
    correctness.append(evaluation.correct)

actual_counts = Counter(record["ctx_size"] for record in records)
assert dict(actual_counts) == EXPECTED_UNIQUE_COUNTS
paper_counts = Counter()
for record in records:
    paper_counts[record["ctx_size"]] += record["paper_weight"]
assert dict(paper_counts) == {length: 600 for length in LENGTHS}
results = model_outputs.append_column(
    "verdict", pa.array(verdicts, type=pa.bool_())
).append_column(
    "correct", pa.array(correctness, type=pa.bool_())
)
{"rows": results.num_rows, "counts_by_length": dict(actual_counts)}

In [ ]:
assert results.num_rows == 9_862
assert RESULT_PATH.name == "results.parquet"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pq.write_table(results, RESULT_PATH, compression="zstd")
frame = results.to_pandas()
RESULT_PATH

In [ ]:
frame["weighted_correct"] = frame["correct"] * frame["paper_weight"]
frame["weighted_missing"] = frame["verdict"].isna() * frame["paper_weight"]
paper_summary = (
    frame.groupby("ctx_size", as_index=False)
    .agg(
        correct=("weighted_correct", "sum"),
        total=("paper_weight", "sum"),
        no_verdict=("weighted_missing", "sum"),
    )
    .sort_values("ctx_size")
)
paper_summary["accuracy"] = paper_summary["correct"] / paper_summary["total"]
assert paper_summary["ctx_size"].tolist() == list(LENGTHS)
assert paper_summary["total"].tolist() == [600] * len(LENGTHS)
display(paper_summary)
plt.figure(figsize=(8, 4.5))
plt.plot(
    paper_summary["ctx_size"],
    paper_summary["accuracy"],
    marker="o",
)
plt.xticks(LENGTHS)
plt.ylim(0, 1)
plt.xlabel("Input length (# nominal tokens)")
plt.ylabel("Accuracy")
plt.title("FLenQA accuracy by input length — paper weighting")
plt.grid(alpha=0.25)
plt.show()

In [ ]:
unique_summary = (
    frame.groupby("ctx_size", as_index=False)
    .agg(
        correct=("correct", "sum"),
        total=("prompt_id", "size"),
        no_verdict=("verdict", lambda values: values.isna().sum()),
    )
    .sort_values("ctx_size")
)
unique_summary["accuracy"] = unique_summary["correct"] / unique_summary["total"]
assert (
    dict(zip(unique_summary["ctx_size"], unique_summary["total"], strict=True))
    == EXPECTED_UNIQUE_COUNTS
)
display(unique_summary)
plt.figure(figsize=(8, 4.5))
plt.plot(
    unique_summary["ctx_size"],
    unique_summary["accuracy"],
    marker="o",
)
plt.xticks(LENGTHS)
plt.ylim(0, 1)
plt.xlabel("Input length (# nominal tokens)")
plt.ylabel("Accuracy")
plt.title("FLenQA accuracy by input length — unique prompts")
plt.grid(alpha=0.25)
plt.show()

In [ ]:
task_summary = (
    frame.groupby(["task", "ctx_size"], as_index=False)
    .agg(correct=("correct", "sum"), total=("prompt_id", "size"))
    .sort_values(["task", "ctx_size"])
)
task_summary["accuracy"] = task_summary["correct"] / task_summary["total"]
display(task_summary)
plt.figure(figsize=(8, 4.5))
for task, task_frame in task_summary.groupby("task", sort=True):
    plt.plot(
        task_frame["ctx_size"],
        task_frame["accuracy"],
        marker="o",
        label=task,
    )
plt.xticks(LENGTHS)
plt.ylim(0, 1)
plt.xlabel("Input length (# nominal tokens)")
plt.ylabel("Accuracy")
plt.title("FLenQA unique-prompt accuracy by task")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

In [ ]:
verdict_labels = (
    frame["verdict"].map({True: "True", False: "False"}).fillna("No verdict")
)
verdict_counts = pd.crosstab(frame["ctx_size"], verdict_labels).reindex(
    index=LENGTHS, columns=["True", "False", "No verdict"], fill_value=0
)
token_lengths = (
    frame.groupby("ctx_size")["n_input_tokens"]
    .agg(["min", "median", "max"])
    .reindex(LENGTHS)
)
display("Verdict counts by nominal length", verdict_counts)
display("Exact Qwen token lengths by nominal bucket", token_lengths)